### Imports

In [ ]:
# For Production
import json
import random
import requests
import numpy as np
import pandas as pd
import plotly.graph_objects as go

In [ ]:
# Colab only
from google.colab import userdata

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

### Global variables

In [ ]:
alphavantage_api = userdata.get('ALPHAVANTAGE_API')
ticker = 'NVDA'

### Functions

In [ ]:
def convert_columns_to_numeric(df, columns_to_exclude=None):
    """
    Converts all columns in a DataFrame to numeric, except for those specified
    in the exclusion list.

    Args:
        df: pandas DataFrame.
        columns_to_exclude: A list of column names to exclude from conversion.
                            Defaults to None (convert all columns).

    Returns:
        pandas DataFrame with specified columns converted to numeric.
    """
    if columns_to_exclude is None:
        columns_to_exclude = []

    # Identify columns to convert to numeric
    columns_to_convert = df.columns.difference(columns_to_exclude)

    # Convert relevant columns to numeric, coercing errors
    for col in columns_to_convert:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    return df

In [ ]:
def calculate_financial_growth(df, columns):
    """
    Calculates the Year-over-Year growth for specified columns in a DataFrame
    using a custom logic for negative previous values.

    Args:
        df: pandas DataFrame with financial statement data.
        columns: A list of column names for which to calculate YoY growth.

    Returns:
        pandas DataFrame with YoY growth columns added for the specified columns.
    """
    df['fiscalDateEnding'] = pd.to_datetime(df['fiscalDateEnding'])
    df_sorted = df.sort_values(by='fiscalDateEnding').reset_index(drop=True)

    # Convert specified columns to numeric, coercing errors
    for col in columns:
        df_sorted[col] = pd.to_numeric(df_sorted[col], errors='coerce')

    # Define the custom Year-over-Year growth calculation function
    def calculate_yoy_growth_custom(current, previous):
        if pd.isna(previous) or previous == 0:
            return None
        if previous < 0:
            return ((current - previous) / abs(previous)) * 100
        else:
            return ((current - previous) / previous) * 100

    # Calculate Year-over-Year growth for each specified column using the custom function
    for col in columns:
        df_sorted[f'previous_{col}'] = df_sorted[col].shift(1)
        df_sorted[f'{col}_YoY_growth%'] = df_sorted.apply(
            lambda row: calculate_yoy_growth_custom(row[col], row[f'previous_{col}']),
            axis=1
        )
        df_sorted = df_sorted.drop(columns=[f'previous_{col}']) # Drop the temporary column

    return df_sorted[['fiscalDateEnding'] + [f'{col}_YoY_growth%' for col in columns]]

In [ ]:
def impute_yoy_median(df):
    """
    Imputes the median for columns containing "YoY" in a DataFrame.

    Args:
        df: pandas DataFrame.

    Returns:
        pandas DataFrame with NaN values in YoY columns imputed with the median.
    """
    yoy_columns = [col for col in df.columns if 'YoY' in col]
    for col in yoy_columns:
        df[col] = df[col].fillna(df[col].median())
    return df

In [ ]:
def convert_columns_to_millions(df, columns_to_convert):
    """
    Converts specified numeric columns in a DataFrame to millions.

    Args:
        df: pandas DataFrame.
        columns_to_convert: A list of column names to convert to millions.

    Returns:
        pandas DataFrame with specified columns converted to millions.
    """
    # Convert specified columns to numeric, coercing errors and then fillna
    for col in columns_to_convert:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # Divide the specified columns by 1,000,000
    for col in columns_to_convert:
        if col in df.columns:
            df[col] = df[col] / 1_000_000

    return df

In [ ]:
def generate_financial_bar_charts(financial_data_transposed: pd.DataFrame):
    """
    Generates Plotly bar charts for financial metrics over time from a transposed DataFrame.

    Args:
        financial_data_transposed: A pandas DataFrame where the index are the metrics
                                   and columns are the years.

    Returns:
        A list of Plotly Figure objects.
    """
    figs = []
    # Convert the index to string type for Plotly compatibility
    financial_data_transposed.index = financial_data_transposed.index.astype(str)

    for index, row in financial_data_transposed.iterrows():
        # Determine the y-axis title based on the index name
        if index == 'EPS':
            yaxis_title = ''
        elif index.endswith('%'):
            yaxis_title = 'Porcentaje'
        else:
            yaxis_title = 'En millones'

        fig = go.Figure()
        # Generate a random color for the bars
        random_color = f'rgb({random.randint(0, 255)}, {random.randint(0, 255)}, {random.randint(0, 255)})'

        # Convert row values to numeric before plotting
        row_values_numeric = pd.to_numeric(row.values, errors='coerce')

        fig.add_trace(go.Bar(x=financial_data_transposed.columns, y=row_values_numeric, name=index, marker_color=random_color))
        fig.update_layout(
            title=index,  # Use the index name as the graph title
            xaxis_title='Year',
            yaxis_title=yaxis_title,
            xaxis=dict(
                tickmode='array',
                tickvals=financial_data_transposed.columns,
                ticktext=financial_data_transposed.columns
            )
        )
        figs.append(fig)

    return figs

## Data Ingestion

### Income Statement (Estado de resultados o P&L)

In [ ]:
url_income = f'https://www.alphavantage.co/query?function=INCOME_STATEMENT&symbol={ticker}&apikey={alphavantage_api}'
response_income = requests.get(url_income)
income_statement = response_income.json()

In [ ]:
income_statement.keys()

dict_keys(['symbol', 'annualReports', 'quarterlyReports'])

In [ ]:
# print(json.dumps(income_statement, indent=4))

In [ ]:
income_df = pd.DataFrame.from_dict(income_statement['annualReports'])
income_df.shape

(20, 26)

In [ ]:
income_df

,fiscalDateEnding,reportedCurrency,grossProfit,totalRevenue,costOfRevenue,costofGoodsAndServicesSold,operatingIncome,sellingGeneralAndAdministrative,researchAndDevelopment,operatingExpenses,investmentIncomeNet,netInterestIncome,interestIncome,interestExpense,nonInterestIncome,otherNonOperatingIncome,depreciation,depreciationAndAmortization,incomeBeforeTax,incomeTaxExpense,interestAndDebtExpense,netIncomeFromContinuingOperations,comprehensiveIncomeNetOfTax,ebit,ebitda,netIncome
0,2026-01-31,USD,153463000000,215938000000,62475000000,62475000000,130387000000,4579000000,18497000000,23076000000,None,2041000000,2300000000,259000000,None,None,None,2843000000,141450000000,21383000000,None,120067000000,None,141709000000,144552000000,120067000000
1,2025-01-31,USD,97858000000,130497000000,32639000000,32639000000,81453000000,3491000000,12914000000,16405000000,None,1539000000,1786000000,247000000,None,None,None,1864000000,84026000000,11146000000,None,72880000000,None,84273000000,86137000000,72880000000
2,2024-01-31,USD,44301000000,60922000000,16621000000,16621000000,32972000000,2654000000,8675000000,11329000000,None,609000000,866000000,257000000,None,None,None,1508000000,33818000000,4058000000,None,29760000000,None,34075000000,35583000000,29760000000
3,2023-01-31,USD,15356000000,26974000000,11618000000,11618000000,4224000000,2440000000,7339000000,11132000000,None,5000000,267000000,262000000,None,219000000,None,1543000000,4181000000,-187000000,None,4368000000,None,4443000000,5986000000,4368000000
4,2022-01-31,USD,17475000000,26914000000,9439000000,9439000000,10041000000,2166000000,5268000000,7434000000,None,-207000000,29000000,236000000,None,136000000,None,1174000000,9941000000,189000000,None,9752000000,None,10177000000,11351000000,9752000000
5,2021-01-31,USD,10396000000,16675000000,6279000000,6279000000,4532000000,1940000000,3924000000,5864000000,None,-127000000,57000000,184000000,None,61000000,None,1098000000,4409000000,77000000,None,4332000000,None,4593000000,5691000000,4332000000
6,2020-01-31,USD,6768000000,10918000000,4150000000,4150000000,2846000000,1093000000,2829000000,3922000000,None,126000000,178000000,52000000,None,176000000,None,381000000,2970000000,174000000,None,2796000000,None,3022000000,3403000000,2796000000
7,2019-01-31,USD,7171000000,11716000000,4545000000,4545000000,3804000000,991000000,2376000000,3367000000,None,78000000,136000000,58000000,None,150000000,None,262000000,3896000000,-245000000,None,4141000000,None,3954000000,4216000000,4141000000
8,2018-01-31,USD,5822000000,9714000000,3892000000,3892000000,3210000000,815000000,1797000000,2612000000,None,8000000,69000000,61000000,None,47000000,None,199000000,3196000000,149000000,None,3047000000,None,3257000000,3456000000,3047000000
9,2017-01-31,USD,4063000000,6910000000,2847000000,2847000000,1934000000,663000000,1463000000,2129000000,None,-4000000,54000000,58000000,None,29000000,None,187000000,1905000000,239000000,None,1666000000,None,1963000000,2150000000,1666000000


In [ ]:
def convert_columns_to_numeric(df, columns_to_exclude=None):
    """
    Converts all columns in a DataFrame to numeric, except for those specified
    in the exclusion list.

    Args:
        df: pandas DataFrame.
        columns_to_exclude: A list of column names to exclude from conversion.
                            Defaults to None (convert all columns).

    Returns:
        pandas DataFrame with specified columns converted to numeric.
    """
    if columns_to_exclude is None:
        columns_to_exclude = []

    # Identify columns to convert to numeric
    columns_to_convert = df.columns.difference(columns_to_exclude)

    # Convert relevant columns to numeric, coercing errors
    for col in columns_to_convert:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    return df

In [ ]:
income_df = convert_columns_to_numeric(income_df, columns_to_exclude=['fiscalDateEnding', 'reportedCurrency'])

In [ ]:
income_df.head()

,fiscalDateEnding,reportedCurrency,grossProfit,totalRevenue,costOfRevenue,costofGoodsAndServicesSold,operatingIncome,sellingGeneralAndAdministrative,researchAndDevelopment,operatingExpenses,investmentIncomeNet,netInterestIncome,interestIncome,interestExpense,nonInterestIncome,otherNonOperatingIncome,depreciation,depreciationAndAmortization,incomeBeforeTax,incomeTaxExpense,interestAndDebtExpense,netIncomeFromContinuingOperations,comprehensiveIncomeNetOfTax,ebit,ebitda,netIncome
0,2026-01-31,USD,153463000000,215938000000,62475000000,62475000000,130387000000,4579000000,18497000000,23076000000,NaN,2041000000.00,2300000000.00,259000000,NaN,NaN,NaN,2843000000,141450000000,21383000000,NaN,120067000000.00,NaN,141709000000,144552000000,120067000000
1,2025-01-31,USD,97858000000,130497000000,32639000000,32639000000,81453000000,3491000000,12914000000,16405000000,NaN,1539000000.00,1786000000.00,247000000,NaN,NaN,NaN,1864000000,84026000000,11146000000,NaN,72880000000.00,NaN,84273000000,86137000000,72880000000
2,2024-01-31,USD,44301000000,60922000000,16621000000,16621000000,32972000000,2654000000,8675000000,11329000000,NaN,609000000.00,866000000.00,257000000,NaN,NaN,NaN,1508000000,33818000000,4058000000,NaN,29760000000.00,NaN,34075000000,35583000000,29760000000
3,2023-01-31,USD,15356000000,26974000000,11618000000,11618000000,4224000000,2440000000,7339000000,11132000000,NaN,5000000.00,267000000.00,262000000,NaN,219000000.00,NaN,1543000000,4181000000,-187000000,NaN,4368000000.00,NaN,4443000000,5986000000,4368000000
4,2022-01-31,USD,17475000000,26914000000,9439000000,9439000000,10041000000,2166000000,5268000000,7434000000,NaN,-207000000.00,29000000.00,236000000,NaN,136000000.00,NaN,1174000000,9941000000,189000000,NaN,9752000000.00,NaN,10177000000,11351000000,9752000000


In [ ]:
pd.DataFrame.from_dict(income_statement['quarterlyReports'])

,fiscalDateEnding,reportedCurrency,grossProfit,totalRevenue,costOfRevenue,costofGoodsAndServicesSold,operatingIncome,sellingGeneralAndAdministrative,researchAndDevelopment,operatingExpenses,investmentIncomeNet,netInterestIncome,interestIncome,interestExpense,nonInterestIncome,otherNonOperatingIncome,depreciation,depreciationAndAmortization,incomeBeforeTax,incomeTaxExpense,interestAndDebtExpense,netIncomeFromContinuingOperations,comprehensiveIncomeNetOfTax,ebit,ebitda,netIncome
0,2026-01-31,USD,51093000000,68127000000,17034000000,17034000000,44299000000,1282000000,5512000000,6794000000,None,495000000,568000000,73000000,None,None,None,812000000,50398000000,7438000000,None,42960000000,None,50471000000,51283000000,42960000000
1,2025-10-31,USD,41849000000,57006000000,15157000000,15157000000,36010000000,1134000000,4705000000,5839000000,None,563000000,624000000,61000000,None,None,None,751000000,37936000000,6026000000,None,31910000000,None,37997000000,38748000000,31910000000
2,2025-07-31,USD,33853000000,46743000000,12890000000,12890000000,28440000000,1122000000,4291000000,5413000000,None,530000000,592000000,62000000,None,None,None,669000000,31206000000,4784000000,None,26422000000,None,31268000000,31937000000,26422000000
3,2025-04-30,USD,26668000000,44062000000,17394000000,17394000000,21638000000,1041000000,3989000000,5030000000,None,452000000,515000000,63000000,None,None,None,611000000,21910000000,3135000000,None,18775000000,None,21973000000,22584000000,18775000000
4,2025-01-31,USD,28723000000,39331000000,10608000000,10608000000,24034000000,975000000,3714000000,4689000000,None,450000000,511000000,61000000,None,None,None,543000000,25217000000,3126000000,None,22091000000,None,25278000000,25821000000,22091000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,2007-01-31,USD,385706000,878873000,493167000,493167000,138514000,84900000,162276000,247192000,None,None,None,4000,None,None,None,34292000,151559000,-11947000,None,0,None,138514000,172806000,163506000
77,2006-10-31,USD,333942000,820572000,486630000,486630000,117613000,69100000,140732000,216329000,None,None,None,22000,None,None,None,25031000,128327000,21816000,None,0,None,117613000,142644000,106511000
78,2006-07-31,USD,292100000,687500000,395400000,395400000,95800000,69100000,127300000,196300000,None,None,None,6000,None,None,None,24200000,104500000,17800000,None,0,None,95800000,120000000,86800000
79,2006-04-30,USD,288757000,681807000,393050000,393050000,100685000,65668000,122404000,188072000,None,None,None,1000,None,None,None,24031000,109248000,18572000,None,0,None,100685000,124716000,90676000


### Balance Sheet (Hoja de balance)

In [ ]:
url_balance = f'https://www.alphavantage.co/query?function=BALANCE_SHEET&symbol={ticker}&apikey={alphavantage_api}'
response_balance = requests.get(url_balance)
balance_sheet = response_balance.json()

In [ ]:
balance_sheet.keys()

dict_keys(['symbol', 'annualReports', 'quarterlyReports'])

In [ ]:
# print(json.dumps(balance_sheet, indent=4))

In [ ]:
balance_df = pd.DataFrame.from_dict(balance_sheet['annualReports'])
balance_df.shape

(20, 38)

In [ ]:
balance_df = convert_columns_to_numeric(balance_df, columns_to_exclude=['fiscalDateEnding', 'reportedCurrency'])

In [ ]:
balance_df.tail()

,fiscalDateEnding,reportedCurrency,totalAssets,totalCurrentAssets,cashAndCashEquivalentsAtCarryingValue,cashAndShortTermInvestments,inventory,currentNetReceivables,totalNonCurrentAssets,propertyPlantEquipment,accumulatedDepreciationAmortizationPPE,intangibleAssets,intangibleAssetsExcludingGoodwill,goodwill,investments,longTermInvestments,shortTermInvestments,otherCurrentAssets,otherNonCurrentAssets,totalLiabilities,totalCurrentLiabilities,currentAccountsPayable,deferredRevenue,currentDebt,shortTermDebt,totalNonCurrentLiabilities,capitalLeaseObligations,longTermDebt,currentLongTermDebt,longTermDebtNoncurrent,shortLongTermDebtTotal,otherCurrentLiabilities,otherNonCurrentLiabilities,totalShareholderEquity,treasuryStock,retainedEarnings,commonStock,commonStockSharesOutstanding
15,2011-01-31,USD,4495246000,3226950000,665361000,665361000,345525000,348770000,1268296000,568857000.00,NaN,243171000,243171000,369844000,NaN,NaN,1825202000,2490563000,NaN,1313784000,942682000,286138000,NaN,NaN,336380000.00,371102000,23389000.00,23389000.00,NaN,NaN,359769000.00,71300000,347713000.00,3181462000,-1479392000.00,2149328000,677000,23547360000
16,2010-01-31,USD,3585918000,2480830000,447221000,447221000,330674000,374963000,1105088000,571858000.00,NaN,120458000,120458000,369844000,NaN,NaN,1281006000,1728227000,NaN,920778000,784378000,344527000,NaN,NaN,390277000.00,136400000,24450000.00,24450000.00,NaN,NaN,414727000.00,29950000,111950000.00,2665140000,-1463268000.00,1896182000,653000,21982960000
17,2009-01-31,USD,3350727000,2167958000,417688000,417688000,537834000,318435000,1182769000,625798000.00,NaN,147101000,147101000,369844000,NaN,NaN,837702000,56299000,NaN,956075000,778591000,218864000,NaN,NaN,NaN,177484000,NaN,25634000.00,NaN,NaN,25634000.00,559727000,NaN,2394652000,-1463268000.00,1964169000,629000,21925040000
18,2008-01-31,USD,3747671000,2888829000,726969000,726969000,358521000,666494000,858842000,359808000.00,NaN,106926000,106926000,354057000,NaN,NaN,1082509000,54336000,NaN,1129759000,967161000,492099000,NaN,NaN,NaN,162598000,NaN,NaN,NaN,NaN,NaN,469206000,NaN,2617912000,-1039632000.00,1994210000,619000,24269280000
19,2007-01-31,USD,2675263000,2031770000,544414000,544414000,354680000,518680000,643493000,260828000.00,NaN,45511000,45511000,301425000,NaN,NaN,573436000,40560000,NaN,668344000,638807000,272075000,NaN,NaN,NaN,29537000,NaN,NaN,NaN,NaN,NaN,365552000,NaN,2006919000,-487120000.00,1196565000,388000,23478500760


### Cashflow Statement (Estado de flujo de efectivo)

In [ ]:
url_cashflow = f'https://www.alphavantage.co/query?function=CASH_FLOW&symbol={ticker}&apikey={alphavantage_api}'
response_cashflow = requests.get(url_cashflow)
cashflow_statement = response_cashflow.json()

In [ ]:
cashflow_statement.keys()

dict_keys(['symbol', 'annualReports', 'quarterlyReports'])

In [ ]:
# print(json.dumps(cashflow_statement, indent=4))

In [ ]:
cashflow_df = pd.DataFrame.from_dict(cashflow_statement['annualReports'])
cashflow_df.shape

(20, 30)

In [ ]:
cashflow_df = convert_columns_to_numeric(cashflow_df, columns_to_exclude=['fiscalDateEnding', 'reportedCurrency'])

In [ ]:
cashflow_df.head()

,fiscalDateEnding,reportedCurrency,operatingCashflow,paymentsForOperatingActivities,proceedsFromOperatingActivities,changeInOperatingLiabilities,changeInOperatingAssets,depreciationDepletionAndAmortization,capitalExpenditures,changeInReceivables,changeInInventory,profitLoss,cashflowFromInvestment,cashflowFromFinancing,proceedsFromRepaymentsOfShortTermDebt,paymentsForRepurchaseOfCommonStock,paymentsForRepurchaseOfEquity,paymentsForRepurchaseOfPreferredStock,dividendPayout,dividendPayoutCommonStock,dividendPayoutPreferredStock,proceedsFromIssuanceOfCommonStock,proceedsFromIssuanceOfLongTermDebtAndCapitalSecuritiesNet,proceedsFromIssuanceOfPreferredStock,proceedsFromRepurchaseOfEquity,proceedsFromSaleOfTreasuryStock,stockBasedCompensation,changeInCashAndCashEquivalents,changeInExchangeRate,netIncome
0,2026-01-31,USD,102718000000,NaN,NaN,NaN,NaN,2843000000,6042000000,NaN,-11324000000,NaN,-52228000000,-48474000000,NaN,NaN,NaN,NaN,974000000.00,974000000.00,NaN,NaN,NaN,NaN,-40086000000,NaN,6386000000,NaN,NaN,120067000000
1,2025-01-31,USD,64089000000,NaN,NaN,NaN,NaN,1864000000,3236000000,NaN,-4781000000,NaN,-20421000000,-42359000000,NaN,NaN,NaN,NaN,834000000.00,834000000.00,NaN,NaN,NaN,NaN,-33706000000,NaN,4737000000,NaN,NaN,72880000000
2,2024-01-31,USD,28090000000,NaN,NaN,NaN,NaN,1508000000,1069000000,NaN,-98000000,NaN,-10566000000,-13633000000,NaN,NaN,NaN,NaN,395000000.00,395000000.00,NaN,NaN,NaN,NaN,-9533000000,NaN,3549000000,NaN,NaN,29760000000
3,2023-01-31,USD,5641000000,NaN,NaN,NaN,NaN,1544000000,1833000000,822000000.00,-2554000000,NaN,7375000000,-11617000000,NaN,NaN,NaN,NaN,398000000.00,398000000.00,NaN,NaN,NaN,NaN,-10039000000,NaN,2709000000,1399000000.00,NaN,4368000000
4,2022-01-31,USD,9108000000,NaN,NaN,NaN,NaN,1174000000,976000000,-2215000000.00,-774000000,NaN,-9830000000,1865000000,NaN,NaN,NaN,NaN,399000000.00,399000000.00,NaN,NaN,NaN,NaN,-1904000000,NaN,2004000000,1143000000.00,NaN,9752000000


### Earnings per share History (Ganancias por acción)

In [ ]:
url_earnings = f'https://www.alphavantage.co/query?function=EARNINGS&symbol={ticker}&apikey={alphavantage_api}'
response_earnings = requests.get(url_earnings)
earnings_history = response_earnings.json()

In [ ]:
earnings_history.keys()

dict_keys(['symbol', 'annualEarnings', 'quarterlyEarnings'])

In [ ]:
# print(json.dumps(earnings_history, indent=4))

In [ ]:
earnings_df = pd.DataFrame.from_dict(earnings_history['annualEarnings'])
earnings_df.shape

(27, 2)

In [ ]:
earnings_df['reportedEPS'] = pd.to_numeric(earnings_df['reportedEPS'], errors='coerce')

In [ ]:
earnings_df

,fiscalDateEnding,reportedEPS
0,2026-01-31,4.78
1,2025-01-31,2.99
2,2024-01-31,1.30
3,2023-01-31,0.33
4,2022-01-31,0.44
5,2021-01-31,0.25
6,2020-01-31,0.15
7,2019-01-31,0.17
8,2018-01-31,0.12
9,2017-01-31,0.06


## Feature Engineering

### Income Statement Feat. Eng.

In [ ]:
# Ensure fiscalDateEnding is in datetime format for both dataframes before merging
income_df['fiscalDateEnding'] = pd.to_datetime(income_df['fiscalDateEnding'])
earnings_df['fiscalDateEnding'] = pd.to_datetime(earnings_df['fiscalDateEnding'])

# Merge the dataframes
income_df = pd.merge(income_df, earnings_df, on='fiscalDateEnding', how='left')

In [ ]:
income_df.head()

,fiscalDateEnding,reportedCurrency,grossProfit,totalRevenue,costOfRevenue,costofGoodsAndServicesSold,operatingIncome,sellingGeneralAndAdministrative,researchAndDevelopment,operatingExpenses,investmentIncomeNet,netInterestIncome,interestIncome,interestExpense,nonInterestIncome,otherNonOperatingIncome,depreciation,depreciationAndAmortization,incomeBeforeTax,incomeTaxExpense,interestAndDebtExpense,netIncomeFromContinuingOperations,comprehensiveIncomeNetOfTax,ebit,ebitda,netIncome,reportedEPS
0,2026-01-31,USD,153463000000,215938000000,62475000000,62475000000,130387000000,4579000000,18497000000,23076000000,NaN,2041000000.00,2300000000.00,259000000,NaN,NaN,NaN,2843000000,141450000000,21383000000,NaN,120067000000.00,NaN,141709000000,144552000000,120067000000,4.78
1,2025-01-31,USD,97858000000,130497000000,32639000000,32639000000,81453000000,3491000000,12914000000,16405000000,NaN,1539000000.00,1786000000.00,247000000,NaN,NaN,NaN,1864000000,84026000000,11146000000,NaN,72880000000.00,NaN,84273000000,86137000000,72880000000,2.99
2,2024-01-31,USD,44301000000,60922000000,16621000000,16621000000,32972000000,2654000000,8675000000,11329000000,NaN,609000000.00,866000000.00,257000000,NaN,NaN,NaN,1508000000,33818000000,4058000000,NaN,29760000000.00,NaN,34075000000,35583000000,29760000000,1.30
3,2023-01-31,USD,15356000000,26974000000,11618000000,11618000000,4224000000,2440000000,7339000000,11132000000,NaN,5000000.00,267000000.00,262000000,NaN,219000000.00,NaN,1543000000,4181000000,-187000000,NaN,4368000000.00,NaN,4443000000,5986000000,4368000000,0.33
4,2022-01-31,USD,17475000000,26914000000,9439000000,9439000000,10041000000,2166000000,5268000000,7434000000,NaN,-207000000.00,29000000.00,236000000,NaN,136000000.00,NaN,1174000000,9941000000,189000000,NaN,9752000000.00,NaN,10177000000,11351000000,9752000000,0.44


In [ ]:
# Ensure fiscalDateEnding is in datetime format for both dataframes before merging
income_df['fiscalDateEnding'] = pd.to_datetime(income_df['fiscalDateEnding'])
balance_df['fiscalDateEnding'] = pd.to_datetime(balance_df['fiscalDateEnding'])

# Merge the dataframes
income_df = pd.merge(income_df, balance_df[['fiscalDateEnding','commonStockSharesOutstanding']], on='fiscalDateEnding', how='left')

In [ ]:
income_df.head()

,fiscalDateEnding,reportedCurrency,grossProfit,totalRevenue,costOfRevenue,costofGoodsAndServicesSold,operatingIncome,sellingGeneralAndAdministrative,researchAndDevelopment,operatingExpenses,investmentIncomeNet,netInterestIncome,interestIncome,interestExpense,nonInterestIncome,otherNonOperatingIncome,depreciation,depreciationAndAmortization,incomeBeforeTax,incomeTaxExpense,interestAndDebtExpense,netIncomeFromContinuingOperations,comprehensiveIncomeNetOfTax,ebit,ebitda,netIncome,reportedEPS,commonStockSharesOutstanding
0,2026-01-31,USD,153463000000,215938000000,62475000000,62475000000,130387000000,4579000000,18497000000,23076000000,NaN,2041000000.00,2300000000.00,259000000,NaN,NaN,NaN,2843000000,141450000000,21383000000,NaN,120067000000.00,NaN,141709000000,144552000000,120067000000,4.78,24432000000
1,2025-01-31,USD,97858000000,130497000000,32639000000,32639000000,81453000000,3491000000,12914000000,16405000000,NaN,1539000000.00,1786000000.00,247000000,NaN,NaN,NaN,1864000000,84026000000,11146000000,NaN,72880000000.00,NaN,84273000000,86137000000,72880000000,2.99,24804000000
2,2024-01-31,USD,44301000000,60922000000,16621000000,16621000000,32972000000,2654000000,8675000000,11329000000,NaN,609000000.00,866000000.00,257000000,NaN,NaN,NaN,1508000000,33818000000,4058000000,NaN,29760000000.00,NaN,34075000000,35583000000,29760000000,1.30,24940000000
3,2023-01-31,USD,15356000000,26974000000,11618000000,11618000000,4224000000,2440000000,7339000000,11132000000,NaN,5000000.00,267000000.00,262000000,NaN,219000000.00,NaN,1543000000,4181000000,-187000000,NaN,4368000000.00,NaN,4443000000,5986000000,4368000000,0.33,25070000000
4,2022-01-31,USD,17475000000,26914000000,9439000000,9439000000,10041000000,2166000000,5268000000,7434000000,NaN,-207000000.00,29000000.00,236000000,NaN,136000000.00,NaN,1174000000,9941000000,189000000,NaN,9752000000.00,NaN,10177000000,11351000000,9752000000,0.44,25350000000


In [ ]:
# Ensure fiscalDateEnding is in datetime format for both dataframes before merging
income_df['fiscalDateEnding'] = pd.to_datetime(income_df['fiscalDateEnding'])
cashflow_df['fiscalDateEnding'] = pd.to_datetime(cashflow_df['fiscalDateEnding'])

# Merge the dataframes
income_df = pd.merge(income_df, cashflow_df[['fiscalDateEnding','depreciationDepletionAndAmortization']], on='fiscalDateEnding', how='left')

In [ ]:
income_df.head()

,fiscalDateEnding,reportedCurrency,grossProfit,totalRevenue,costOfRevenue,costofGoodsAndServicesSold,operatingIncome,sellingGeneralAndAdministrative,researchAndDevelopment,operatingExpenses,investmentIncomeNet,netInterestIncome,interestIncome,interestExpense,nonInterestIncome,otherNonOperatingIncome,depreciation,depreciationAndAmortization,incomeBeforeTax,incomeTaxExpense,interestAndDebtExpense,netIncomeFromContinuingOperations,comprehensiveIncomeNetOfTax,ebit,ebitda,netIncome,reportedEPS,commonStockSharesOutstanding,depreciationDepletionAndAmortization
0,2026-01-31,USD,153463000000,215938000000,62475000000,62475000000,130387000000,4579000000,18497000000,23076000000,NaN,2041000000.00,2300000000.00,259000000,NaN,NaN,NaN,2843000000,141450000000,21383000000,NaN,120067000000.00,NaN,141709000000,144552000000,120067000000,4.78,24432000000,2843000000
1,2025-01-31,USD,97858000000,130497000000,32639000000,32639000000,81453000000,3491000000,12914000000,16405000000,NaN,1539000000.00,1786000000.00,247000000,NaN,NaN,NaN,1864000000,84026000000,11146000000,NaN,72880000000.00,NaN,84273000000,86137000000,72880000000,2.99,24804000000,1864000000
2,2024-01-31,USD,44301000000,60922000000,16621000000,16621000000,32972000000,2654000000,8675000000,11329000000,NaN,609000000.00,866000000.00,257000000,NaN,NaN,NaN,1508000000,33818000000,4058000000,NaN,29760000000.00,NaN,34075000000,35583000000,29760000000,1.30,24940000000,1508000000
3,2023-01-31,USD,15356000000,26974000000,11618000000,11618000000,4224000000,2440000000,7339000000,11132000000,NaN,5000000.00,267000000.00,262000000,NaN,219000000.00,NaN,1543000000,4181000000,-187000000,NaN,4368000000.00,NaN,4443000000,5986000000,4368000000,0.33,25070000000,1544000000
4,2022-01-31,USD,17475000000,26914000000,9439000000,9439000000,10041000000,2166000000,5268000000,7434000000,NaN,-207000000.00,29000000.00,236000000,NaN,136000000.00,NaN,1174000000,9941000000,189000000,NaN,9752000000.00,NaN,10177000000,11351000000,9752000000,0.44,25350000000,1174000000


In [ ]:
# EBITDA Margin
income_df['ebitdaMargin%'] = income_df['ebitda']*100 / income_df['totalRevenue']
# EBIT Margin
income_df['ebitMargin%'] = income_df['operatingIncome']*100 / income_df['totalRevenue']
# Net income Margin
income_df['netIncomeMargin%'] = income_df['netIncome']*100 / income_df['totalRevenue']
# Net Interest
income_df['netInterest'] = income_df['interestIncome'] - income_df['interestExpense']
# Tax Rate
income_df['taxRate%'] = income_df['incomeTaxExpense']*100 / income_df['incomeBeforeTax']

In [ ]:
income_df.head()

,fiscalDateEnding,reportedCurrency,grossProfit,totalRevenue,costOfRevenue,costofGoodsAndServicesSold,operatingIncome,sellingGeneralAndAdministrative,researchAndDevelopment,operatingExpenses,investmentIncomeNet,netInterestIncome,interestIncome,interestExpense,nonInterestIncome,otherNonOperatingIncome,depreciation,depreciationAndAmortization,incomeBeforeTax,incomeTaxExpense,interestAndDebtExpense,netIncomeFromContinuingOperations,comprehensiveIncomeNetOfTax,ebit,ebitda,netIncome,reportedEPS,commonStockSharesOutstanding,depreciationDepletionAndAmortization,ebitdaMargin%,ebitMargin%,netIncomeMargin%,netInterest,taxRate%
0,2026-01-31,USD,153463000000,215938000000,62475000000,62475000000,130387000000,4579000000,18497000000,23076000000,NaN,2041000000.00,2300000000.00,259000000,NaN,NaN,NaN,2843000000,141450000000,21383000000,NaN,120067000000.00,NaN,141709000000,144552000000,120067000000,4.78,24432000000,2843000000,66.94,60.38,55.60,2041000000.00,15.12
1,2025-01-31,USD,97858000000,130497000000,32639000000,32639000000,81453000000,3491000000,12914000000,16405000000,NaN,1539000000.00,1786000000.00,247000000,NaN,NaN,NaN,1864000000,84026000000,11146000000,NaN,72880000000.00,NaN,84273000000,86137000000,72880000000,2.99,24804000000,1864000000,66.01,62.42,55.85,1539000000.00,13.26
2,2024-01-31,USD,44301000000,60922000000,16621000000,16621000000,32972000000,2654000000,8675000000,11329000000,NaN,609000000.00,866000000.00,257000000,NaN,NaN,NaN,1508000000,33818000000,4058000000,NaN,29760000000.00,NaN,34075000000,35583000000,29760000000,1.30,24940000000,1508000000,58.41,54.12,48.85,609000000.00,12.00
3,2023-01-31,USD,15356000000,26974000000,11618000000,11618000000,4224000000,2440000000,7339000000,11132000000,NaN,5000000.00,267000000.00,262000000,NaN,219000000.00,NaN,1543000000,4181000000,-187000000,NaN,4368000000.00,NaN,4443000000,5986000000,4368000000,0.33,25070000000,1544000000,22.19,15.66,16.19,5000000.00,-4.47
4,2022-01-31,USD,17475000000,26914000000,9439000000,9439000000,10041000000,2166000000,5268000000,7434000000,NaN,-207000000.00,29000000.00,236000000,NaN,136000000.00,NaN,1174000000,9941000000,189000000,NaN,9752000000.00,NaN,10177000000,11351000000,9752000000,0.44,25350000000,1174000000,42.18,37.31,36.23,-207000000.00,1.90


In [ ]:
income_df.tail()

,fiscalDateEnding,reportedCurrency,grossProfit,totalRevenue,costOfRevenue,costofGoodsAndServicesSold,operatingIncome,sellingGeneralAndAdministrative,researchAndDevelopment,operatingExpenses,investmentIncomeNet,netInterestIncome,interestIncome,interestExpense,nonInterestIncome,otherNonOperatingIncome,depreciation,depreciationAndAmortization,incomeBeforeTax,incomeTaxExpense,interestAndDebtExpense,netIncomeFromContinuingOperations,comprehensiveIncomeNetOfTax,ebit,ebitda,netIncome,reportedEPS,commonStockSharesOutstanding,depreciationDepletionAndAmortization,ebitdaMargin%,ebitMargin%,netIncomeMargin%,netInterest,taxRate%
15,2011-01-31,USD,1409090000,3543309000,2134219000,2134219000,255747000,362000000,848830000,1153343000,NaN,NaN,NaN,3127000,NaN,18549000.00,NaN,186989000,271169000,18023000,NaN,253146000.00,NaN,255747000,442736000,253146000,0.02,23547360000,186989000,12.49,7.22,7.14,NaN,6.65
16,2010-01-31,USD,1176923000,3326445000,2149522000,2149522000,-98945000,367017000,908851000,1275868000,NaN,NaN,NaN,3320000,NaN,19971000.00,NaN,196664000,-82294000,-14307000,NaN,-67987000.00,NaN,-78974000,117690000,-67987000,0.01,21982960000,196664000,3.54,-2.97,-2.04,NaN,17.39
17,2009-01-31,USD,1174269000,3424859000,2250590000,2250590000,-70700000,362000000,855879000,1244969000,NaN,NaN,NaN,406000,NaN,NaN,NaN,185023000,-42954000,-12913000,NaN,NaN,NaN,-42548000,142475000,-30041000,0.01,21925040000,185023000,4.16,-2.06,-0.88,NaN,30.06
18,2008-01-31,USD,1869280000,4097860000,2228580000,2228580000,836346000,341000000,691637000,1032934000,NaN,NaN,NaN,0,NaN,NaN,NaN,133192000,901341000,103696000,NaN,NaN,NaN,836346000,969538000,797645000,0.04,24269280000,133192000,23.66,20.41,19.46,NaN,11.50
19,2007-01-31,USD,1300449000,3068771000,1768322000,1768322000,453452000,294000000,553467000,846997000,NaN,NaN,NaN,21000,NaN,NaN,NaN,107562000,494480000,46350000,NaN,NaN,NaN,453452000,561014000,448834000,0.02,23478500760,107562000,18.28,14.78,14.63,NaN,9.37


In [ ]:
income_df['reportedEPS'] = income_df['reportedEPS'].fillna(0)

In [ ]:
income_df['interestIncome'] = income_df['interestIncome'].fillna(0)
income_df['netInterest'] = income_df['interestIncome'] - income_df['interestExpense']

In [ ]:
income_columns_to_grow = ['totalRevenue','grossProfit','operatingIncome','ebitda',
                          'netIncome','reportedEPS','commonStockSharesOutstanding']
income_df_with_growth = calculate_financial_growth(income_df.copy(), income_columns_to_grow)

In [ ]:
income_df_with_growth.head()

,fiscalDateEnding,totalRevenue_YoY_growth%,grossProfit_YoY_growth%,operatingIncome_YoY_growth%,ebitda_YoY_growth%,netIncome_YoY_growth%,reportedEPS_YoY_growth%,commonStockSharesOutstanding_YoY_growth%
0,2007-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2008-01-31,33.53,43.74,84.44,72.82,77.71,57.26,3.37
2,2009-01-31,-16.42,-37.18,-108.45,-85.30,-103.77,-66.67,-9.66
3,2010-01-31,-2.87,0.23,-39.95,-17.40,-126.31,-15.38,0.26
4,2011-01-31,6.52,19.73,358.47,276.19,472.34,54.55,7.12


In [ ]:
# Merge the two dataframes on 'fiscalDateEnding'
income_df = pd.merge(income_df, income_df_with_growth, on='fiscalDateEnding', how='left')

In [ ]:
income_df.tail()

,fiscalDateEnding,reportedCurrency,grossProfit,totalRevenue,costOfRevenue,costofGoodsAndServicesSold,operatingIncome,sellingGeneralAndAdministrative,researchAndDevelopment,operatingExpenses,investmentIncomeNet,netInterestIncome,interestIncome,interestExpense,nonInterestIncome,otherNonOperatingIncome,depreciation,depreciationAndAmortization,incomeBeforeTax,incomeTaxExpense,interestAndDebtExpense,netIncomeFromContinuingOperations,comprehensiveIncomeNetOfTax,ebit,ebitda,netIncome,reportedEPS,commonStockSharesOutstanding,depreciationDepletionAndAmortization,ebitdaMargin%,ebitMargin%,netIncomeMargin%,netInterest,taxRate%,totalRevenue_YoY_growth%,grossProfit_YoY_growth%,operatingIncome_YoY_growth%,ebitda_YoY_growth%,netIncome_YoY_growth%,reportedEPS_YoY_growth%,commonStockSharesOutstanding_YoY_growth%
15,2011-01-31,USD,1409090000,3543309000,2134219000,2134219000,255747000,362000000,848830000,1153343000,NaN,NaN,0.00,3127000,NaN,18549000.00,NaN,186989000,271169000,18023000,NaN,253146000.00,NaN,255747000,442736000,253146000,0.02,23547360000,186989000,12.49,7.22,7.14,-3127000.00,6.65,6.52,19.73,358.47,276.19,472.34,54.55,7.12
16,2010-01-31,USD,1176923000,3326445000,2149522000,2149522000,-98945000,367017000,908851000,1275868000,NaN,NaN,0.00,3320000,NaN,19971000.00,NaN,196664000,-82294000,-14307000,NaN,-67987000.00,NaN,-78974000,117690000,-67987000,0.01,21982960000,196664000,3.54,-2.97,-2.04,-3320000.00,17.39,-2.87,0.23,-39.95,-17.40,-126.31,-15.38,0.26
17,2009-01-31,USD,1174269000,3424859000,2250590000,2250590000,-70700000,362000000,855879000,1244969000,NaN,NaN,0.00,406000,NaN,NaN,NaN,185023000,-42954000,-12913000,NaN,NaN,NaN,-42548000,142475000,-30041000,0.01,21925040000,185023000,4.16,-2.06,-0.88,-406000.00,30.06,-16.42,-37.18,-108.45,-85.30,-103.77,-66.67,-9.66
18,2008-01-31,USD,1869280000,4097860000,2228580000,2228580000,836346000,341000000,691637000,1032934000,NaN,NaN,0.00,0,NaN,NaN,NaN,133192000,901341000,103696000,NaN,NaN,NaN,836346000,969538000,797645000,0.04,24269280000,133192000,23.66,20.41,19.46,0.00,11.50,33.53,43.74,84.44,72.82,77.71,57.26,3.37
19,2007-01-31,USD,1300449000,3068771000,1768322000,1768322000,453452000,294000000,553467000,846997000,NaN,NaN,0.00,21000,NaN,NaN,NaN,107562000,494480000,46350000,NaN,NaN,NaN,453452000,561014000,448834000,0.02,23478500760,107562000,18.28,14.78,14.63,-21000.00,9.37,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
keep_income_cols = ['fiscalDateEnding','totalRevenue','totalRevenue_YoY_growth%',
                    'ebitda','ebitdaMargin%','ebitda_YoY_growth%','depreciationDepletionAndAmortization',
                    'operatingIncome','ebitMargin%','operatingIncome_YoY_growth%',
                    'interestExpense','interestIncome','netInterest',
                    'incomeBeforeTax','incomeTaxExpense','taxRate%',
                    'netIncome','netIncomeMargin%','netIncome_YoY_growth%',
                    'reportedEPS','reportedEPS_YoY_growth%',
                    'commonStockSharesOutstanding','commonStockSharesOutstanding_YoY_growth%']

In [ ]:
income_df = income_df[keep_income_cols].copy()

In [ ]:
income_df['reportedEPS_YoY_growth%'] = income_df['reportedEPS_YoY_growth%'].fillna(0)

In [ ]:
income_df = impute_yoy_median(income_df)

In [ ]:
income_df

,fiscalDateEnding,totalRevenue,totalRevenue_YoY_growth%,ebitda,ebitdaMargin%,ebitda_YoY_growth%,depreciationDepletionAndAmortization,operatingIncome,ebitMargin%,operatingIncome_YoY_growth%,interestExpense,interestIncome,netInterest,incomeBeforeTax,incomeTaxExpense,taxRate%,netIncome,netIncomeMargin%,netIncome_YoY_growth%,reportedEPS,reportedEPS_YoY_growth%,commonStockSharesOutstanding,commonStockSharesOutstanding_YoY_growth%
0,2026-01-31,215938000000,65.47,144552000000,66.94,67.82,2843000000,130387000000,60.38,60.08,259000000,2300000000.00,2041000000.00,141450000000,21383000000,15.12,120067000000,55.60,64.75,4.78,59.76,24432000000,-1.50
1,2025-01-31,130497000000,114.20,86137000000,66.01,142.07,1864000000,81453000000,62.42,147.04,247000000,1786000000.00,1539000000.00,84026000000,11146000000,13.26,72880000000,55.85,144.89,2.99,130.69,24804000000,-0.55
2,2024-01-31,60922000000,125.85,35583000000,58.41,494.44,1508000000,32972000000,54.12,680.59,257000000,866000000.00,609000000.00,33818000000,4058000000,12.00,29760000000,48.85,581.32,1.30,289.49,24940000000,-0.52
3,2023-01-31,26974000000,0.22,5986000000,22.19,-47.26,1544000000,4224000000,15.66,-57.93,262000000,267000000.00,5000000.00,4181000000,-187000000,-4.47,4368000000,16.19,-55.21,0.33,-25.08,25070000000,-1.10
4,2022-01-31,26914000000,61.40,11351000000,42.18,99.46,1174000000,10041000000,37.31,121.56,236000000,29000000.00,-207000000.00,9941000000,189000000,1.90,9752000000,36.23,125.12,0.44,77.45,25350000000,0.92
5,2021-01-31,16675000000,52.73,5691000000,34.13,67.23,1098000000,4532000000,27.18,59.24,184000000,57000000.00,-127000000.00,4409000000,77000000,1.75,4332000000,25.98,54.94,0.25,72.40,25120000000,1.62
6,2020-01-31,10918000000,-6.81,3403000000,31.17,-19.28,381000000,2846000000,26.07,-25.18,52000000,178000000.00,126000000.00,2970000000,174000000,5.86,2796000000,25.61,-32.48,0.15,-12.47,24720000000,-1.12
7,2019-01-31,11716000000,20.61,4216000000,35.98,21.99,262000000,3804000000,32.47,18.50,58000000,136000000.00,78000000.00,3896000000,-245000000,-6.29,4141000000,35.34,35.90,0.17,44.35,25000000000,-1.11
8,2018-01-31,9714000000,40.58,3456000000,35.58,60.74,199000000,3210000000,33.05,65.98,61000000,69000000.00,8000000.00,3196000000,149000000,4.66,3047000000,31.37,82.89,0.12,79.69,25280000000,-2.62
9,2017-01-31,6910000000,37.92,2150000000,31.11,117.83,187000000,1934000000,27.99,158.90,58000000,54000000.00,-4000000.00,1905000000,239000000,12.55,1666000000,24.11,171.34,0.06,113.33,25960000000,14.06


### FCF Feat. Eng.

In [ ]:
cashflow_df.head()

,fiscalDateEnding,reportedCurrency,operatingCashflow,paymentsForOperatingActivities,proceedsFromOperatingActivities,changeInOperatingLiabilities,changeInOperatingAssets,depreciationDepletionAndAmortization,capitalExpenditures,changeInReceivables,changeInInventory,profitLoss,cashflowFromInvestment,cashflowFromFinancing,proceedsFromRepaymentsOfShortTermDebt,paymentsForRepurchaseOfCommonStock,paymentsForRepurchaseOfEquity,paymentsForRepurchaseOfPreferredStock,dividendPayout,dividendPayoutCommonStock,dividendPayoutPreferredStock,proceedsFromIssuanceOfCommonStock,proceedsFromIssuanceOfLongTermDebtAndCapitalSecuritiesNet,proceedsFromIssuanceOfPreferredStock,proceedsFromRepurchaseOfEquity,proceedsFromSaleOfTreasuryStock,stockBasedCompensation,changeInCashAndCashEquivalents,changeInExchangeRate,netIncome
0,2026-01-31,USD,102718000000,NaN,NaN,NaN,NaN,2843000000,6042000000,NaN,-11324000000,NaN,-52228000000,-48474000000,NaN,NaN,NaN,NaN,974000000.00,974000000.00,NaN,NaN,NaN,NaN,-40086000000,NaN,6386000000,NaN,NaN,120067000000
1,2025-01-31,USD,64089000000,NaN,NaN,NaN,NaN,1864000000,3236000000,NaN,-4781000000,NaN,-20421000000,-42359000000,NaN,NaN,NaN,NaN,834000000.00,834000000.00,NaN,NaN,NaN,NaN,-33706000000,NaN,4737000000,NaN,NaN,72880000000
2,2024-01-31,USD,28090000000,NaN,NaN,NaN,NaN,1508000000,1069000000,NaN,-98000000,NaN,-10566000000,-13633000000,NaN,NaN,NaN,NaN,395000000.00,395000000.00,NaN,NaN,NaN,NaN,-9533000000,NaN,3549000000,NaN,NaN,29760000000
3,2023-01-31,USD,5641000000,NaN,NaN,NaN,NaN,1544000000,1833000000,822000000.00,-2554000000,NaN,7375000000,-11617000000,NaN,NaN,NaN,NaN,398000000.00,398000000.00,NaN,NaN,NaN,NaN,-10039000000,NaN,2709000000,1399000000.00,NaN,4368000000
4,2022-01-31,USD,9108000000,NaN,NaN,NaN,NaN,1174000000,976000000,-2215000000.00,-774000000,NaN,-9830000000,1865000000,NaN,NaN,NaN,NaN,399000000.00,399000000.00,NaN,NaN,NaN,NaN,-1904000000,NaN,2004000000,1143000000.00,NaN,9752000000


In [ ]:
fcf_df = cashflow_df[['fiscalDateEnding','operatingCashflow','depreciationDepletionAndAmortization','capitalExpenditures']].copy()

In [ ]:
fcf_df['maintenanceCapex'] = fcf_df.apply(
    lambda row: row['capitalExpenditures'] if row['capitalExpenditures'] < row['depreciationDepletionAndAmortization'] else row['depreciationDepletionAndAmortization'],
    axis=1
)

In [ ]:
fcf_df['growthCapex'] = fcf_df['capitalExpenditures'] - fcf_df['maintenanceCapex']

In [ ]:
fcf_df.head()

,fiscalDateEnding,operatingCashflow,depreciationDepletionAndAmortization,capitalExpenditures,maintenanceCapex,growthCapex
0,2026-01-31,102718000000,2843000000,6042000000,2843000000,3199000000
1,2025-01-31,64089000000,1864000000,3236000000,1864000000,1372000000
2,2024-01-31,28090000000,1508000000,1069000000,1069000000,0
3,2023-01-31,5641000000,1544000000,1833000000,1544000000,289000000
4,2022-01-31,9108000000,1174000000,976000000,976000000,0


In [ ]:
# Ensure fiscalDateEnding is in datetime format for both dataframes before merging
fcf_df['fiscalDateEnding'] = pd.to_datetime(fcf_df['fiscalDateEnding'])
income_df['fiscalDateEnding'] = pd.to_datetime(income_df['fiscalDateEnding'])

# Merge the dataframes
fcf_df = pd.merge(fcf_df, income_df[['fiscalDateEnding','totalRevenue','operatingIncome','interestExpense','incomeTaxExpense']], on='fiscalDateEnding', how='left')

In [ ]:
fcf_df.head()

,fiscalDateEnding,operatingCashflow,depreciationDepletionAndAmortization,capitalExpenditures,maintenanceCapex,growthCapex,totalRevenue,operatingIncome,interestExpense,incomeTaxExpense
0,2026-01-31,102718000000,2843000000,6042000000,2843000000,3199000000,215938000000,130387000000,259000000,21383000000
1,2025-01-31,64089000000,1864000000,3236000000,1864000000,1372000000,130497000000,81453000000,247000000,11146000000
2,2024-01-31,28090000000,1508000000,1069000000,1069000000,0,60922000000,32972000000,257000000,4058000000
3,2023-01-31,5641000000,1544000000,1833000000,1544000000,289000000,26974000000,4224000000,262000000,-187000000
4,2022-01-31,9108000000,1174000000,976000000,976000000,0,26914000000,10041000000,236000000,189000000


In [ ]:
# Ensure fiscalDateEnding is in datetime format for both dataframes before merging
fcf_df['fiscalDateEnding'] = pd.to_datetime(fcf_df['fiscalDateEnding'])
balance_df['fiscalDateEnding'] = pd.to_datetime(balance_df['fiscalDateEnding'])

# Merge the dataframes
fcf_df = pd.merge(fcf_df, balance_df[['fiscalDateEnding','inventory','currentNetReceivables',
                                      'currentAccountsPayable','deferredRevenue',
                                      'otherNonCurrentLiabilities','commonStockSharesOutstanding']], on='fiscalDateEnding', how='left')

In [ ]:
fcf_df = fcf_df.fillna(0)

# Convert float columns to integers
float_cols = fcf_df.select_dtypes(include='float64').columns
for col in float_cols:
    fcf_df[col] = fcf_df[col].astype(int)

In [ ]:
fcf_df.head()

,fiscalDateEnding,operatingCashflow,depreciationDepletionAndAmortization,capitalExpenditures,maintenanceCapex,growthCapex,totalRevenue,operatingIncome,interestExpense,incomeTaxExpense,inventory,currentNetReceivables,currentAccountsPayable,deferredRevenue,otherNonCurrentLiabilities,commonStockSharesOutstanding
0,2026-01-31,102718000000,2843000000,6042000000,2843000000,3199000000,215938000000,130387000000,259000000,21383000000,21403000000,38466000000,9812000000,0,381000000,24432000000
1,2025-01-31,64089000000,1864000000,3236000000,1864000000,1372000000,130497000000,81453000000,247000000,11146000000,10080000000,23065000000,6310000000,0,79000000,24804000000
2,2024-01-31,28090000000,1508000000,1069000000,1069000000,0,60922000000,32972000000,257000000,4058000000,5282000000,9999000000,2699000000,0,65000000,24940000000
3,2023-01-31,5641000000,1544000000,1833000000,1544000000,289000000,26974000000,4224000000,262000000,-187000000,5159000000,3827000000,1193000000,0,1913000000,25070000000
4,2022-01-31,9108000000,1174000000,976000000,976000000,0,26914000000,10041000000,236000000,189000000,2605000000,4650000000,1783000000,0,1553000000,25350000000


In [ ]:
# Replace 'deferredRevenue' with 'otherNonCurrentLiabilities' if 'deferredRevenue' is not positive
fcf_df['deferredRevenue'] = fcf_df.apply(
    lambda row: row['otherNonCurrentLiabilities'] if row['deferredRevenue'] < 0 else row['deferredRevenue'],
    axis=1
)

# Drop the 'otherNonCurrentLiabilities' column
fcf_df = fcf_df.drop(columns=['otherNonCurrentLiabilities'])

In [ ]:
fcf_df.head()

,fiscalDateEnding,operatingCashflow,depreciationDepletionAndAmortization,capitalExpenditures,maintenanceCapex,growthCapex,totalRevenue,operatingIncome,interestExpense,incomeTaxExpense,inventory,currentNetReceivables,currentAccountsPayable,deferredRevenue,commonStockSharesOutstanding
0,2026-01-31,102718000000,2843000000,6042000000,2843000000,3199000000,215938000000,130387000000,259000000,21383000000,21403000000,38466000000,9812000000,0,24432000000
1,2025-01-31,64089000000,1864000000,3236000000,1864000000,1372000000,130497000000,81453000000,247000000,11146000000,10080000000,23065000000,6310000000,0,24804000000
2,2024-01-31,28090000000,1508000000,1069000000,1069000000,0,60922000000,32972000000,257000000,4058000000,5282000000,9999000000,2699000000,0,24940000000
3,2023-01-31,5641000000,1544000000,1833000000,1544000000,289000000,26974000000,4224000000,262000000,-187000000,5159000000,3827000000,1193000000,0,25070000000
4,2022-01-31,9108000000,1174000000,976000000,976000000,0,26914000000,10041000000,236000000,189000000,2605000000,4650000000,1783000000,0,25350000000


In [ ]:
# Impute the median for 'deferredRevenue' if necessary
if fcf_df['deferredRevenue'].isnull().any():
    median_deferredRevenue = fcf_df['deferredRevenue'].median()
    fcf_df['deferredRevenue'] = fcf_df['deferredRevenue'].fillna(median_deferredRevenue)

In [ ]:
fcf_df['ebitda'] = fcf_df['operatingIncome'] + fcf_df['depreciationDepletionAndAmortization']

In [ ]:
fcf_df['workingCapital'] = fcf_df['inventory'] + fcf_df['currentNetReceivables'] - fcf_df['currentAccountsPayable'] - fcf_df['deferredRevenue']

In [ ]:
fcf_df.head()

,fiscalDateEnding,operatingCashflow,depreciationDepletionAndAmortization,capitalExpenditures,maintenanceCapex,growthCapex,totalRevenue,operatingIncome,interestExpense,incomeTaxExpense,inventory,currentNetReceivables,currentAccountsPayable,deferredRevenue,commonStockSharesOutstanding,ebitda,workingCapital
0,2026-01-31,102718000000,2843000000,6042000000,2843000000,3199000000,215938000000,130387000000,259000000,21383000000,21403000000,38466000000,9812000000,0,24432000000,133230000000,50057000000
1,2025-01-31,64089000000,1864000000,3236000000,1864000000,1372000000,130497000000,81453000000,247000000,11146000000,10080000000,23065000000,6310000000,0,24804000000,83317000000,26835000000
2,2024-01-31,28090000000,1508000000,1069000000,1069000000,0,60922000000,32972000000,257000000,4058000000,5282000000,9999000000,2699000000,0,24940000000,34480000000,12582000000
3,2023-01-31,5641000000,1544000000,1833000000,1544000000,289000000,26974000000,4224000000,262000000,-187000000,5159000000,3827000000,1193000000,0,25070000000,5768000000,7793000000
4,2022-01-31,9108000000,1174000000,976000000,976000000,0,26914000000,10041000000,236000000,189000000,2605000000,4650000000,1783000000,0,25350000000,11215000000,5472000000


In [ ]:
fcf_df['previous_currentAccountsPayable'] = fcf_df['currentAccountsPayable'].shift(-1)
fcf_df['previous_workingCapital'] = fcf_df['workingCapital'].shift(-1)

fcf_df['changeInWC'] = fcf_df.apply(lambda row: (row['workingCapital'] - row['previous_workingCapital']) if row['previous_currentAccountsPayable'] > 0 else 0, axis=1)

# Drop the temporary columns
fcf_df = fcf_df.drop(columns=['previous_currentAccountsPayable', 'previous_workingCapital'])

In [ ]:
fcf_df.head()

,fiscalDateEnding,operatingCashflow,depreciationDepletionAndAmortization,capitalExpenditures,maintenanceCapex,growthCapex,totalRevenue,operatingIncome,interestExpense,incomeTaxExpense,inventory,currentNetReceivables,currentAccountsPayable,deferredRevenue,commonStockSharesOutstanding,ebitda,workingCapital,changeInWC
0,2026-01-31,102718000000,2843000000,6042000000,2843000000,3199000000,215938000000,130387000000,259000000,21383000000,21403000000,38466000000,9812000000,0,24432000000,133230000000,50057000000,23222000000.00
1,2025-01-31,64089000000,1864000000,3236000000,1864000000,1372000000,130497000000,81453000000,247000000,11146000000,10080000000,23065000000,6310000000,0,24804000000,83317000000,26835000000,14253000000.00
2,2024-01-31,28090000000,1508000000,1069000000,1069000000,0,60922000000,32972000000,257000000,4058000000,5282000000,9999000000,2699000000,0,24940000000,34480000000,12582000000,4789000000.00
3,2023-01-31,5641000000,1544000000,1833000000,1544000000,289000000,26974000000,4224000000,262000000,-187000000,5159000000,3827000000,1193000000,0,25070000000,5768000000,7793000000,2321000000.00
4,2022-01-31,9108000000,1174000000,976000000,976000000,0,26914000000,10041000000,236000000,189000000,2605000000,4650000000,1783000000,0,25350000000,11215000000,5472000000,2418000000.00


In [ ]:
fcf_df['freeCashFlow'] = fcf_df['ebitda'] - fcf_df['interestExpense'] - fcf_df['incomeTaxExpense'] - fcf_df['maintenanceCapex'] - fcf_df['changeInWC']
fcf_df['freeCashFlowPerShare'] = fcf_df['freeCashFlow'] / fcf_df['commonStockSharesOutstanding']

In [ ]:
fcf_df['freeCashFlow2'] = fcf_df['operatingCashflow'] - fcf_df['capitalExpenditures']

In [ ]:
fcf_df['freeCashFlow'] - fcf_df['freeCashFlow2']

,0
0,-11153000000.00
1,-5046000000.00
2,-2714000000.00
3,-1980000000.00
4,-736000000.00
5,-1528000000.00
6,-1113000000.00
7,-175000000.00
8,-239000000.00
9,-356000000.00


In [ ]:
fcf_df.head()

,fiscalDateEnding,operatingCashflow,depreciationDepletionAndAmortization,capitalExpenditures,maintenanceCapex,growthCapex,totalRevenue,operatingIncome,interestExpense,incomeTaxExpense,inventory,currentNetReceivables,currentAccountsPayable,deferredRevenue,commonStockSharesOutstanding,ebitda,workingCapital,changeInWC,freeCashFlow,freeCashFlowPerShare,freeCashFlow2
0,2026-01-31,102718000000,2843000000,6042000000,2843000000,3199000000,215938000000,130387000000,259000000,21383000000,21403000000,38466000000,9812000000,0,24432000000,133230000000,50057000000,23222000000.00,85523000000.00,3.50,96676000000
1,2025-01-31,64089000000,1864000000,3236000000,1864000000,1372000000,130497000000,81453000000,247000000,11146000000,10080000000,23065000000,6310000000,0,24804000000,83317000000,26835000000,14253000000.00,55807000000.00,2.25,60853000000
2,2024-01-31,28090000000,1508000000,1069000000,1069000000,0,60922000000,32972000000,257000000,4058000000,5282000000,9999000000,2699000000,0,24940000000,34480000000,12582000000,4789000000.00,24307000000.00,0.97,27021000000
3,2023-01-31,5641000000,1544000000,1833000000,1544000000,289000000,26974000000,4224000000,262000000,-187000000,5159000000,3827000000,1193000000,0,25070000000,5768000000,7793000000,2321000000.00,1828000000.00,0.07,3808000000
4,2022-01-31,9108000000,1174000000,976000000,976000000,0,26914000000,10041000000,236000000,189000000,2605000000,4650000000,1783000000,0,25350000000,11215000000,5472000000,2418000000.00,7396000000.00,0.29,8132000000


In [ ]:
fcf_df_with_growth = calculate_financial_growth(fcf_df.copy(), ['freeCashFlow','freeCashFlowPerShare'])

In [ ]:
fcf_df_with_growth.tail()

,fiscalDateEnding,freeCashFlow_YoY_growth%,freeCashFlowPerShare_YoY_growth%
15,2022-01-31,133.61,131.49
16,2023-01-31,-75.28,-75.01
17,2024-01-31,1229.70,1236.64
18,2025-01-31,129.59,130.85
19,2026-01-31,53.25,55.58


In [ ]:
# Merge the two dataframes on 'fiscalDateEnding'
fcf_df = pd.merge(fcf_df, fcf_df_with_growth, on='fiscalDateEnding', how='left')

In [ ]:
fcf_df.tail()

,fiscalDateEnding,operatingCashflow,depreciationDepletionAndAmortization,capitalExpenditures,maintenanceCapex,growthCapex,totalRevenue,operatingIncome,interestExpense,incomeTaxExpense,inventory,currentNetReceivables,currentAccountsPayable,deferredRevenue,commonStockSharesOutstanding,ebitda,workingCapital,changeInWC,freeCashFlow,freeCashFlowPerShare,freeCashFlow2,freeCashFlow_YoY_growth%,freeCashFlowPerShare_YoY_growth%
15,2011-01-31,675797000,186989000,97890000,97890000,0,3543309000,255747000,3127000,18023000,345525000,348770000,286138000,0,23547360000,442736000,408157000,47047000.00,276649000.00,0.01,577907000,-10.00,-15.98
16,2010-01-31,487807000,196664000,77601000,77601000,0,3326445000,-98945000,3320000,-14307000,330674000,374963000,344527000,0,21982960000,97719000,361110000,-276295000.00,307400000.00,0.01,410206000,288.96,288.46
17,2009-01-31,249360000,185023000,407670000,185023000,222647000,3424859000,-70700000,406000,-12913000,537834000,318435000,218864000,0,21925040000,114323000,637405000,104489000.00,-162682000.00,-0.01,-158310000,-120.31,-122.48
18,2008-01-31,1270196000,133192000,187745000,133192000,54553000,4097860000,836346000,0,103696000,358521000,666494000,492099000,0,24269280000,969538000,532916000,-68369000.00,801019000.00,0.03,1082451000,96.77,90.36
19,2007-01-31,587111000,107562000,145256000,107562000,37694000,3068771000,453452000,21000,46350000,354680000,518680000,272075000,0,23478500760,561014000,601285000,0.00,407081000.00,0.02,441855000,NaN,NaN


In [ ]:
# Calculate the requested ratios and add them to the fcf_df DataFrame
fcf_df['maintenanceCapexMargin%'] = fcf_df['maintenanceCapex']*100 / fcf_df['totalRevenue']
fcf_df['workingCapitalMargin%'] = fcf_df['workingCapital']*100 / fcf_df['totalRevenue']
fcf_df['freeCashFlowMargin%'] = fcf_df['freeCashFlow']*100 / fcf_df['totalRevenue']
fcf_df['cashConversion%'] = fcf_df['freeCashFlow']*100 / fcf_df['ebitda']

In [ ]:
keep_fcf_cols = ['fiscalDateEnding','ebitda','capitalExpenditures',
       'maintenanceCapex', 'growthCapex','interestExpense',
       'incomeTaxExpense','inventory','currentNetReceivables', 'currentAccountsPayable',
       'deferredRevenue','workingCapital','changeInWC', 'freeCashFlow',
       'freeCashFlow_YoY_growth%','freeCashFlowPerShare',
       'freeCashFlowPerShare_YoY_growth%','maintenanceCapexMargin%',
       'workingCapitalMargin%','freeCashFlowMargin%','cashConversion%']

In [ ]:
fcf_df = fcf_df[keep_fcf_cols]

In [ ]:
fcf_df.tail()

,fiscalDateEnding,ebitda,capitalExpenditures,maintenanceCapex,growthCapex,interestExpense,incomeTaxExpense,inventory,currentNetReceivables,currentAccountsPayable,deferredRevenue,workingCapital,changeInWC,freeCashFlow,freeCashFlow_YoY_growth%,freeCashFlowPerShare,freeCashFlowPerShare_YoY_growth%,maintenanceCapexMargin%,workingCapitalMargin%,freeCashFlowMargin%,cashConversion%
15,2011-01-31,442736000,97890000,97890000,0,3127000,18023000,345525000,348770000,286138000,0,408157000,47047000.00,276649000.00,-10.00,0.01,-15.98,2.76,11.52,7.81,62.49
16,2010-01-31,97719000,77601000,77601000,0,3320000,-14307000,330674000,374963000,344527000,0,361110000,-276295000.00,307400000.00,288.96,0.01,288.46,2.33,10.86,9.24,314.58
17,2009-01-31,114323000,407670000,185023000,222647000,406000,-12913000,537834000,318435000,218864000,0,637405000,104489000.00,-162682000.00,-120.31,-0.01,-122.48,5.40,18.61,-4.75,-142.30
18,2008-01-31,969538000,187745000,133192000,54553000,0,103696000,358521000,666494000,492099000,0,532916000,-68369000.00,801019000.00,96.77,0.03,90.36,3.25,13.00,19.55,82.62
19,2007-01-31,561014000,145256000,107562000,37694000,21000,46350000,354680000,518680000,272075000,0,601285000,0.00,407081000.00,NaN,0.02,NaN,3.51,19.59,13.27,72.56


In [ ]:
fcf_df = impute_yoy_median(fcf_df)

In [ ]:
fcf_df

,fiscalDateEnding,ebitda,capitalExpenditures,maintenanceCapex,growthCapex,interestExpense,incomeTaxExpense,inventory,currentNetReceivables,currentAccountsPayable,deferredRevenue,workingCapital,changeInWC,freeCashFlow,freeCashFlow_YoY_growth%,freeCashFlowPerShare,freeCashFlowPerShare_YoY_growth%,maintenanceCapexMargin%,workingCapitalMargin%,freeCashFlowMargin%,cashConversion%
0,2026-01-31,133230000000,6042000000,2843000000,3199000000,259000000,21383000000,21403000000,38466000000,9812000000,0,50057000000,23222000000.00,85523000000.00,53.25,3.50,55.58,1.32,23.18,39.61,64.19
1,2025-01-31,83317000000,3236000000,1864000000,1372000000,247000000,11146000000,10080000000,23065000000,6310000000,0,26835000000,14253000000.00,55807000000.00,129.59,2.25,130.85,1.43,20.56,42.76,66.98
2,2024-01-31,34480000000,1069000000,1069000000,0,257000000,4058000000,5282000000,9999000000,2699000000,0,12582000000,4789000000.00,24307000000.00,1229.70,0.97,1236.64,1.75,20.65,39.90,70.50
3,2023-01-31,5768000000,1833000000,1544000000,289000000,262000000,-187000000,5159000000,3827000000,1193000000,0,7793000000,2321000000.00,1828000000.00,-75.28,0.07,-75.01,5.72,28.89,6.78,31.69
4,2022-01-31,11215000000,976000000,976000000,0,236000000,189000000,2605000000,4650000000,1783000000,0,5472000000,2418000000.00,7396000000.00,133.61,0.29,131.49,3.63,20.33,27.48,65.95
5,2021-01-31,5630000000,1128000000,1098000000,30000000,184000000,77000000,1826000000,2429000000,1201000000,0,3054000000,1105000000.00,3166000000.00,0.22,0.13,-1.37,6.58,18.31,18.99,56.23
6,2020-01-31,3227000000,489000000,381000000,108000000,52000000,174000000,979000000,1657000000,687000000,0,1949000000,-539000000.00,3159000000.00,6.44,0.13,7.64,3.49,17.85,28.93,97.89
7,2019-01-31,4066000000,600000000,262000000,338000000,58000000,-245000000,1575000000,1424000000,511000000,0,2488000000,1023000000.00,2968000000.00,11.16,0.12,12.41,2.24,21.24,25.33,73.00
8,2018-01-31,3409000000,593000000,199000000,394000000,61000000,149000000,796000000,1265000000,596000000,0,1465000000,330000000.00,2670000000.00,134.21,0.11,140.51,2.05,15.08,27.49,78.32
9,2017-01-31,2121000000,176000000,176000000,0,58000000,239000000,794000000,826000000,485000000,0,1135000000,508000000.00,1140000000.00,58.71,0.04,39.14,2.55,16.43,16.50,53.75
